# 05 — Metric Building & Mart Construction

Phase 4 pipeline: build mart tables, compute KPIs, export app-layer CSVs for Tableau.

**Outputs:**
- `mart_panel_year.parquet` — review panel × year aggregations with normalized KPIs
- `mart_product_code_year.parquet` — product code × year aggregations
- `mart_firm_product_year.parquet` — manufacturer × product code × year with firm share
- `data/app/*.csv` — Tableau-ready CSV exports

In [ ]:
import sys

sys.path.insert(0, "..")


from src.marts.builder import build_all_marts
from src.marts.export import export_all
from src.qa.summary import build_qa_summary, evaluate_quality_gate

## 1. Build All Marts

In [ ]:
marts = build_all_marts()

for name, df in marts.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} cols")

## 2. Validate Grain Uniqueness

In [ ]:
grain_keys = {
    "mart_panel_year": ["review_panel", "year"],
    "mart_product_code_year": ["product_code", "year"],
    "mart_firm_product_year": ["manufacturer", "product_code", "year"],
}

for name, keys in grain_keys.items():
    df = marts[name]
    dups = df.duplicated(subset=keys).sum()
    assert dups == 0, f"{name} has {dups} duplicate keys!"
    print(f"{name}: grain unique on {keys} ✓")

## 3. Spot-Check Key Panels

In [ ]:
panel_df = marts["mart_panel_year"]

for panel in ["AN", "CV", "OR", "SU", "HO"]:
    rows = panel_df[panel_df["review_panel"] == panel]
    if rows.empty:
        print(f"\n{panel}: no data")
        continue
    print(f"\n=== Panel {panel} ===")
    cols = [
        "year",
        "event_count_dedup",
        "recall_count",
        "clearance_count",
        "events_per_100_clearances",
        "severe_recall_share",
    ]
    display_cols = [c for c in cols if c in rows.columns]
    print(rows[display_cols].to_string(index=False))

## 4. Verify NULL Thresholds

In [ ]:
from src.marts.kpis import MIN_CLEARANCES_FOR_NORM, MIN_RECALLS_FOR_SHARE

# Panel mart: events_per_100_clearances should be NULL when clearances < threshold
null_kpi = panel_df[panel_df["events_per_100_clearances"].isna()]
print(f"Rows with NULL events_per_100_clearances: {len(null_kpi)}")
if not null_kpi.empty:
    below = null_kpi["clearance_count"] < MIN_CLEARANCES_FOR_NORM
    print(f"  All have clearance_count < {MIN_CLEARANCES_FOR_NORM}: {below.all()}")

# severe_recall_share should be NULL when total recalls < threshold
null_srs = panel_df[panel_df["severe_recall_share"].isna()]
print(f"\nRows with NULL severe_recall_share: {len(null_srs)}")
if not null_srs.empty:
    below_r = null_srs["recall_count"] < MIN_RECALLS_FOR_SHARE
    print(f"  All have recall_count < {MIN_RECALLS_FOR_SHARE}: {below_r.all()}")

## 5. Export App CSVs

In [ ]:
app_tables = export_all()

limits = {
    "app_overview": 5_000,
    "app_category_product": 20_000,
    "app_manufacturer": 30_000,
    "app_methodology": None,
}

for name, df in app_tables.items():
    limit = limits.get(name)
    status = ""
    if limit:
        status = f" ({'OK' if len(df) <= limit else 'OVER LIMIT'}, max {limit:,})"
    print(f"{name}: {len(df):,} rows{status}")

## 6. QA Summary & Quality Gate

In [ ]:
qa_df = build_qa_summary()

qa_df.style.map(
    lambda v: (
        "background-color: #d4edda"
        if v == "pass"
        else ("background-color: #fff3cd" if v == "warn" else ("background-color: #f8d7da" if v == "fail" else ""))
    ),
    subset=["status"],
)

In [ ]:
gate_pass = evaluate_quality_gate(qa_df)
fail_count = (qa_df["status"] == "fail").sum()
warn_count = (qa_df["status"] == "warn").sum()

print(f"Checks: {len(qa_df)} total, {fail_count} fail, {warn_count} warn")
print(f"\n{'=' * 40}")
print(f"QUALITY GATE: {'PASS — proceed to Phase 5' if gate_pass else 'FAIL — review issues above'}")

## 7. Visualizations

In [ ]:
import matplotlib.pyplot as plt

# Top panels by total deduped events
panel_totals = panel_df.groupby("review_panel")["event_count_dedup"].sum().nlargest(15)

fig, ax = plt.subplots(figsize=(10, 5))
panel_totals.plot.barh(ax=ax)
ax.set_xlabel("Total Deduped Events")
ax.set_ylabel("Review Panel")
ax.set_title("Top 15 Review Panels by Adverse Event Volume")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: events vs clearances per product code (latest year)
pc_df = marts["mart_product_code_year"]
latest_year = pc_df["year"].max()
latest = pc_df[pc_df["year"] == latest_year].copy()

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(latest["clearance_count"], latest["event_count_dedup"], alpha=0.4, s=15)
ax.set_xlabel("Clearance Count")
ax.set_ylabel("Deduped Event Count")
ax.set_title(f"Events vs Clearances by Product Code ({latest_year})")
plt.tight_layout()
plt.show()